In [1]:
import pandas as pd
import numpy as np
import os

from transformers import pipeline
from tqdm import tqdm

In [2]:
import sys
sys.path.append("..")

In [3]:
books = pd.read_csv("../data/books_with_categories.csv")

print("Dataset shape:", books.shape)

books.head()

Dataset shape: (5197, 14)


,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description,simple_categories
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Fiction
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Fiction


In [4]:
classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
    device="cpu",
    batch_size=64
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
emotion_labels = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise",
    "neutral"
]

In [6]:
def split_into_sentences(text):

    if pd.isna(text):
        return []

    sentences = [
        s.strip()
        for s in text.split(".")
        if len(s.strip()) > 20
    ]

    return sentences

In [7]:
def calculate_max_emotion_scores(predictions):

    per_emotion_scores = {label: [] for label in emotion_labels}

    for prediction in predictions:

        sorted_predictions = sorted(
            prediction,
            key=lambda x: x["label"]
        )

        for index, label in enumerate(emotion_labels):

            per_emotion_scores[label].append(
                sorted_predictions[index]["score"]
            )

    return {
        label: np.max(scores)
        for label, scores in per_emotion_scores.items()
    }

In [8]:
cache_file = "../cache/emotion_scores.csv"

In [9]:
if os.path.exists(cache_file):

    emotions_df = pd.read_csv(cache_file)

    print("Loaded cached emotion scores")

else:

    emotions_df = None

In [10]:
if emotions_df is None:

    isbn = []
    emotion_scores = {label: [] for label in emotion_labels}

    for i in tqdm(range(len(books))):

        isbn.append(books["isbn13"][i])

        sentences = split_into_sentences(
            books["description"][i]
        )

        if len(sentences) == 0:

            for label in emotion_labels:
                emotion_scores[label].append(0)

            continue

        predictions = classifier(sentences)

        max_scores = calculate_max_emotion_scores(predictions)

        for label in emotion_labels:
            emotion_scores[label].append(max_scores[label])

    emotions_df = pd.DataFrame(emotion_scores)

    emotions_df["isbn13"] = isbn

    emotions_df.to_csv(cache_file, index=False)

100%|██████████| 5197/5197 [14:51<00:00,  5.83it/s]  


In [11]:
books = pd.merge(
    books,
    emotions_df,
    on="isbn13"
)

books.head()

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,...,title_and_subtitle,tagged_description,simple_categories,anger,disgust,fear,joy,sadness,surprise,neutral
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,...,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction,0.029641,0.338239,0.983973,0.949027,0.697845,0.956065,0.729602
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,...,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction,0.594468,0.461990,0.935215,0.704422,0.891110,0.051413,0.212222
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,...,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction,0.041301,0.024568,0.973285,0.767238,0.042176,0.010860,0.009796
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,...,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Fiction,0.325306,0.125763,0.436338,0.242209,0.732685,0.043272,0.029084
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,...,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Fiction,0.091732,0.197434,0.095043,0.041146,0.890048,0.475880,0.012260


In [12]:
books.to_csv(
    "../data/books_with_emotions.csv",
    index=False
)